# Agent 输出模式 (LangGraph)

使用 `create_react_agent` 创建的 Agent 支持多种输出模式：
- **values**: 返回完整的状态值
- **updates**: 返回每次更新的增量
- **messages**: 返回消息流
- **debug**: 返回调试信息
- **custom**: 自定义输出模式

**注意**: `invoke()` 始终返回最终状态，流式输出模式需要用 `stream()` 方法

In [7]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 创建 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 定义工具
@tool
def get_weather(city: str) -> str:
    """查询城市天气"""
    weather = {"北京": "晴 25°C", "上海": "多云 22°C", "广州": "雨 28°C"}
    return weather.get(city, f"暂无{city}天气数据")

@tool  
def calculate(expression: str) -> str:
    """计算数学表达式"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"计算错误: {e}"

# 创建 Agent (使用新 API)
agent = create_agent(llm, [get_weather, calculate])
print("Agent 创建成功!")

Agent 创建成功!


## 1. invoke() - 返回最终状态 (默认 values 模式)

In [8]:
# invoke() 始终返回最终完整状态
result = agent.invoke({"messages": [("human", "北京天气怎么样?")]})

rprint("[bold green]=== invoke() 输出 ===[/bold green]")
rprint(f"状态键: {list(result.keys())}")
rprint(f"消息数量: {len(result['messages'])}")
for msg in result["messages"]:
    content = msg.content[:60] if msg.content else "(空)"
    rprint(f"  - {msg.type}: {content}")

=== invoke() 输出 ===

状态键: ['messages']

消息数量: 4

- human: 北京天气怎么样?

- ai: (空)

- tool: 晴 25°C

- ai: 北京现在天气晴朗，气温25°C，是个不错的天气呢！☀️

## 2. stream(stream_mode="values") - 流式返回完整状态

In [11]:
# values 模式：每个步骤返回完整状态
rprint("[bold cyan]=== values 模式 ===[/bold cyan]")

for i, state in enumerate(
    agent.stream(
        {"messages": [("human", "计算 3+5*2")]},
        stream_mode="values"
    )
):
    rprint(f"\n[bold]步骤 {i+1}:[/bold]")
    last_msg = state["messages"][-1]
    rprint(f"  消息类型: {last_msg.type}")
    rprint(f"  内容: {last_msg.content[:80] if last_msg.content else '(空)'}")

=== values 模式 ===

步骤 1:

消息类型: human

内容: 计算 3+5*2

步骤 2:

...

消息类型: tool

内容: 13

步骤 4:

消息类型: ai

内容: 计算结果是 **13**。

这是因为根据数学运算顺序，需要先计算乘法（5×2=10），然后再进行加法（3+10=13）。

## 3. stream(stream_mode="updates") - 返回增量更新

In [12]:
# updates 模式：返回每次节点执行后的状态变化
rprint("[bold yellow]=== updates 模式 ===[/bold yellow]")

for i, update in enumerate(
    agent.stream(
        {"messages": [("human", "计算 3+5*2")]},
        stream_mode="updates"
    )
):
    rprint(f"\n[bold]更新 {i+1}:[/bold]")
    rprint(f"  节点: {list(update.keys())}")
    
    if "agent" in update:
        msg = update["agent"]["messages"][-1]
        rprint(f"  Agent 决策: {msg.content[:80] if msg.content else '调用工具'}")
    if "tools" in update:
        msg = update["tools"]["messages"][-1]
        rprint(f"  工具结果: {msg.content[:80]}")

=== updates 模式 ===

更新 1:

节点: ['model']

更新 2:

节点: ['tools']

工具结果: 13

更新 3:

节点: ['model']

## 4. stream(stream_mode="messages") - 返回消息流

In [ ]:
# messages 模式：流式返回 token 级别的消息
rprint("[bold magenta]=== messages 模式 ===[/bold magenta]")

for token, metadata in agent.stream(
    {"messages": [("human", "用一句话介绍北京")]},
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)
print()

## 5. stream(stream_mode="debug") - 返回调试信息

In [ ]:
# debug 模式：返回详细的执行过程
rprint("[bold red]=== debug 模式 ===[/bold red]")

for i, event in enumerate(
    agent.stream(
        {"messages": [("human", "计算 10*20")]},
        stream_mode="debug"
    )
):
    rprint(f"\n[bold]事件 {i+1}:[/bold]")
    rprint(f"  类型: {event.get('type', 'N/A')}")
    if 'payload' in event:
        rprint(f"  详情: {str(event['payload'])[:100]}")

## 6. stream(stream_mode="custom") - 自定义输出

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# 自定义状态类型
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    tool_calls_count: int
    final_answer: str

# 自定义 Agent 节点
def custom_agent(state: AgentState):
    messages = state["messages"]
    response = llm.invoke(messages)
    return {
        "messages": [response],
        "tool_calls_count": state.get("tool_calls_count", 0) + len(response.tool_calls),
        "final_answer": response.content if not response.tool_calls else ""
    }

# 自定义工具节点
def custom_tool_node(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    
    results = []
    for tc in last_message.tool_calls:
        if tc["name"] == "get_weather":
            result = get_weather.invoke(tc["args"])
        elif tc["name"] == "calculate":
            result = calculate.invoke(tc["args"])
        else:
            result = "未知工具"
        results.append({"role": "tool", "content": result, "tool_call_id": tc["id"]})
    
    return {"messages": results}

# 构建自定义图
workflow = StateGraph(AgentState)
workflow.add_node("agent", custom_agent)
workflow.add_node("tools", custom_tool_node)
workflow.set_entry_point("agent")

def should_continue(state: AgentState):
    last_msg = state["messages"][-1]
    return "tools" if last_msg.tool_calls else END

workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

custom_agent_graph = workflow.compile()

# 使用 stream 获取自定义状态
rprint("[bold blue]=== custom 模式 ===[/bold blue]")

for state in custom_agent_graph.stream(
    {"messages": [("human", "北京天气怎么样?")], "tool_calls_count": 0, "final_answer": ""},
    stream_mode="values"
):
    rprint(f"工具调用次数: {state.get('tool_calls_count', 0)}")
    rprint(f"最终回答: {state.get('final_answer', '处理中...')}")
    rprint("---")

=== custom 模式 ===

工具调用次数: 0

最终回答:

---

工具调用次数: 0

最终回答: 关于北京的天气，我目前无法提供实时数据，但可以为您补充一些实用信息：

1. **实时查询建议**  
   您可以使用手机自带的天气App（如苹果天气、小米天气）、搜索引擎（百度/搜狗搜索“北京天气”），或访问中国天气网（weat
her.com.cn）获取最新预报。

2. **典型气候参考**  
   - **春季（3-5月）**：多风沙，昼夜温差大  
   - **夏季（6-8月）**：高温多雨，最高温可达35℃+  
   - **秋季（9-11月）**：秋高气爽，10月平均气温约15℃  
   - **冬季（12-2月）**：寒冷干燥，常有雾霾，1月平均气温-3℃左右  

3. **出行小提示**  
   若近期前往北京，建议携带外套（防温差）、雨具（夏季多雷阵雨），冬季需注意防寒保暖。

需要具体某天的天气详情，可以随时告诉我日期，我会尽力为您整合公开气象信息参考 😊

---

: 

## 总结

| 方法 | stream_mode | 说明 | 返回内容 |
|------|-------------|------|----------|
| `invoke()` | - | 执行并返回最终状态 | 完整状态字典 |
| `stream()` | `values` | 每步返回完整状态 | 状态字典序列 |
| `stream()` | `updates` | 每步返回增量更新 | 节点名→更新内容 |
| `stream()` | `messages` | 流式返回 token | (token, metadata) |
| `stream()` | `debug` | 返回调试事件 | 事件详情 |
| `stream()` | `custom` | 自定义图输出 | 自定义状态 |